In [0]:
# Live 4 (O) — Orquestração
# Notebook: 20_gold_vendas_receita_mensal_produto_uf
# Objetivo: Gold de receita mensal por UF/produto (mesma definição da Live 2).

from pyspark.sql import functions as F

dbutils.widgets.text("tgt_schema", "aulas_ao_vivo.live_20260323")
dbutils.widgets.text("valid_status", "pago")

TGT_SCHEMA = dbutils.widgets.get("tgt_schema")
VALID_STATUS = dbutils.widgets.get("valid_status")

TBL_SILVER_ITENS = f"{TGT_SCHEMA}.silver_vendas_itens_pedido"
TBL_GOLD = f"{TGT_SCHEMA}.gold_vendas_receita_mensal_produto_uf"

itens = spark.table(TBL_SILVER_ITENS)
base_gold = itens.filter(F.col("status") == F.lit(VALID_STATUS))

gold = (
    base_gold
    .groupBy("mes_referencia", "uf", "produto")
    .agg(
        F.round(F.sum("valor_total"), 2).alias("receita_total"),
        F.countDistinct("id_pedido").alias("qtd_pedidos_validos"),
        F.round(F.avg("valor_total"), 2).alias("ticket_medio_item")
    )
    .orderBy("mes_referencia", "uf", "produto")
)

gold.write.mode("overwrite").format("delta").saveAsTable(TBL_GOLD)

out = spark.table(TBL_GOLD)
out.printSchema()
display(out.orderBy(F.col("mes_referencia").asc(), F.col("uf").asc(), F.col("receita_total").desc()).limit(50))
